In [8]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
import re
import os
print("Libraries Imported")

Libraries Imported


In [5]:
from utils import format_signal, format_flow, format_sleep_profile

In [16]:
#returned a list
events = format_flow("Data/AP01/flow_events.txt")
events[0:3]

[{'start': Timestamp('2024-05-30 23:48:45.119000'),
  'end': Timestamp('2024-05-30 23:49:01.408000'),
  'duration': 16,
  'label': 'Hypopnea',
  'stage': 'N1'},
 {'start': Timestamp('2024-05-30 23:50:16.578000'),
  'end': Timestamp('2024-05-30 23:50:33.546000'),
  'duration': 17,
  'label': 'Hypopnea',
  'stage': 'N1'},
 {'start': Timestamp('2024-05-30 23:52:13.626000'),
  'end': Timestamp('2024-05-30 23:52:27.268000'),
  'duration': 14,
  'label': 'Hypopnea',
  'stage': 'N1'}]

In [ ]:
subject_folder=input("Enter the Subject folder path")
if not os.path.exists(subject_folder):
        print("Folder does not exist")
else:
    flow_df=format_signal(f"{subject_folder}/nasal_airflow.txt")
    thorac_df=format_signal(f"{subject_folder}/thoracic_movement.txt")
    spo2_df=format_signal(f"{subject_folder}/spo2.txt")
    events=format_flow(f"{subject_folder}/flow_events.txt") #dict
    sleep_df=format_sleep_profile(f"{subject_folder}/sleep_profile.txt")

    print("All files are loaded!")

In [4]:
'''returns a df
spo2_df=format_signal("Data/AP01/spo2.txt")
spo2_df.head()'''

'returns a df\nspo2_df=format_signal("Data/AP01/spo2.txt")\nspo2_df.head()'

In [ ]:
def plot_signal(ax, df, color, ylabel, title): #signal
    t0=df.index[0]
    hours=(df.index-t0).total_seconds()/3600

    ax.plot(
        hours,
        df["value"].values,
        color=color,
        linewidth=0.5,
        alpha=0.9
    )

    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3, linewidth=0.4)

    return t0, hours

In [2]:
#flow_events
event_colours={
    "Hypopnea" : "#FDFF00", #yellow
    "Apnea" : "#FF8400" #orange
}

def shade_event(ax, events, t0): #flow_events
    for e in events:
        start = (ev["start"]-t0).total_seconds()/3600
        end= (ev["end"]-t0).total_seconds()/3600
        colour=event_colours.get(ev["label"],"#00FF2F") #green as default for unknown
        ax.axvspan(
            start, end,
            alpha=0.2,
            color=colour
        )

In [27]:
u_stages = set(item['stage'] for item in events)
print(u_stages)

{'N3', 'N2', 'N1', 'Wake', 'REM'}


In [28]:
sleep_df =format_sleep_profile("Data/AP01/sleep_profile.txt")
print(sleep_df["stage"].unique())

['Wake' 'N1' 'N2' 'N3' 'REM']


In [1]:
#sleep_profiles
#pastel colours sleep profile
stage_colours={ 
    "Wake":"#FF775C",
    "REM":"#FFB65C",
    "N1": "#BDFF91",
    "N2": "#8AFFF3",
    "N3":"#8A9DFF"
}

def plot_sleep_profile(px, sleep_df, t0):
    stages=events["stage"].values #np
    t=sleep_df.index
    h=(t-t0).total_seconds()/3600
    s=len(stages)
    for i in range(s):
        colour=stage_colours.get(stages[i], "#AAAAAA") #default grey
        ax.barh(
            0,                          
            hours[i+1] - hours[i],
            left=hours[i],
            color=colour,
            height=1,
            align="center"
        )
    px.set_ylabel("Sleep Stage", fontsize=8)
    px.set_title("Sleep Stages", fontsize=10, fontweight="bold")
    px.tick_params(labelsize=7)

    # ── Add legend ──
    # create a colored box for each sleep stage
    patches = [
        mpatches.Patch(color=colour, label=stages)
        for stage, coluor in stage_colours.items()
    ]

    # add legend to plot
    px.legend(
        handles=patches,
        loc="upper right",
        fontsize=6,
        ncol=4,
        framealpha=0.7
    )